# Day A5 — 표 다루기와 그림: base R, dplyr, ggplot2

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/A-r-bioconductor/day-A5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/A-r-bioconductor/day-A5.md)

## 예시로 이해하기

> 모든 값은 연습용 **가상 값**입니다. 세 예시가 같은 가상 데이터를 씁니다. 먼저 아래 코드를 `a5_data.R`로 저장해 두고 각 예시 맨 위에서 `source("a5_data.R")`로 불러옵니다(같은 폴더에서 실행한다고 가정).

In [ ]:
# 레슨처럼 아래 코드를 a5_data.R 파일로 저장한 뒤 불러온다 (뒤 셀들이 source()로 다시 부른다)
writeLines(r"---(
# a5_data.R — 유전자 4개 × 샘플 8개의 long 형식 가상 데이터
set.seed(42)
samples <- data.frame(sample = sprintf("S%02d", 1:8),
                      group  = rep(c("ctrl", "treat"), each = 4),
                      batch  = rep(c("b1", "b2"), times = 4))
genes <- c("Gfap", "Aif1", "Cd68", "Tmem119")
shift <- c(Gfap = 1.0, Aif1 = 0.6, Cd68 = 1.5, Tmem119 = 0)
long <- expand.grid(gene = genes, sample = samples$sample, stringsAsFactors = FALSE)
long <- merge(long, samples, by = "sample")
long$log2expr <- round(8 + rnorm(nrow(long), sd = 0.4) +
                       ifelse(long$group == "treat", shift[long$gene], 0), 2)
)---", "a5_data.R")
source("a5_data.R")

### 예시 1 — base R로 요약하고 모양 바꾸기

In [ ]:
source("a5_data.R")
head(long, 4)
# base R
agg <- aggregate(log2expr ~ gene + group, data = long, FUN = mean)
agg[order(agg$gene), ]
wide <- reshape(long[, c("gene", "sample", "log2expr")], idvar = "gene",
                timevar = "sample", direction = "wide")
dim(wide)

### 예시 2 — dplyr/tidyr 파이프라인

In [ ]:
source("a5_data.R")
suppressPackageStartupMessages({ library(dplyr); library(tidyr) })
summ <- long |>
  filter(gene != "Tmem119") |>
  group_by(gene, group) |>
  summarise(mean = mean(log2expr), n = n(), .groups = "drop") |>
  pivot_wider(names_from = group, values_from = mean) |>
  mutate(diff = round(treat - ctrl, 2)) |>
  arrange(desc(diff))
summ
mat <- long |>
  select(gene, sample, log2expr) |>
  pivot_wider(names_from = sample, values_from = log2expr)
dim(mat)

### 예시 3 — ggplot2로 군 비교 그림 저장

In [ ]:
source("a5_data.R")
suppressPackageStartupMessages(library(ggplot2))
dir.create("figures/advanced/A", recursive = TRUE, showWarnings = FALSE)
p <- ggplot(long, aes(x = group, y = log2expr)) +
  geom_boxplot(outlier.shape = NA, fill = "grey90") +
  geom_jitter(aes(shape = batch), width = 0.1, size = 2) +
  facet_wrap(~ gene, nrow = 1) +
  labs(x = NULL, y = "log2 정규화 발현(가상)", title = "가상 데이터: 유전자별 군 비교") +
  theme_bw(base_size = 11)
ggsave("figures/advanced/A/A5_gene_boxplots.png", p, width = 8, height = 3, dpi = 150)
file.exists("figures/advanced/A/A5_gene_boxplots.png")
class(p)